# Deletion Ladder + Depth-Ramped Noise

**Follow-up to "Deep-Bit Surgery".** Deleting the deepest 10% of tree bits changed nothing. The professor asked for two things next: **(1)** delete at more levels, **(2)** noise whose strength grows with depth.

**One fact that shapes this notebook:** almost half of all bits sit in the bottom layer (depth 5 = 2,464 of 5,399 bits). The 'deepest 10%' was only a fifth of that layer, and 20% is still inside it. So we also delete **whole layers**.

### Part A — deletion ladder (12 arms): *where does the signal live?*
| ordering | fractions removed |
|---|---|
| **deepest first** | 10%, 20%, layer 5 (46%), layers 4-5 (73%), layers 3-5 (87%) |
| **random** (same sizes) | the matched control: if random = deep, depth doesn't matter — the encoding is just redundant |
| **shallowest** (reverse) | delete only layers 0-2 (13%) — if this hurts while deep 13% doesn't, sharpest confirmation |

### Part B — depth-ramped bit flips (4 arms), bits stay binary
`p(depth) = P × depth/5` → roots never flip, bottom layer flips with P. Uniform-flip controls carry the **same average amount of noise**, so 'does the *shape* matter' is separated from 'does the *amount* matter'.
| arm | bottom-layer p | average p |
|---|---|---|
| ramp 0.2 | 0.20 | 0.16 |
| ramp 0.5 | 0.50 | 0.40 |
| uniform 0.16 | 0.16 | 0.16 |
| uniform 0.40 | 0.40 | 0.40 |

Everything else identical to the last two experiments: credit, x+tree, OOB-honest, hard bits, 2-member ensembles, 400 epochs, dropout/L1 off.

⏱ **~2 hours on an A100** (16 arms × ~7 min). Parts A and B are separate cells — run either. Runtime → GPU → Run all.

In [ ]:
# 1 · GPU check
import torch
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU -> Runtime > GPU')

In [ ]:
# 2 · get the code
%cd /content
!git clone https://github.com/sushanedulloo/TKCE.git 2>/dev/null || echo 'already cloned'
%cd /content/TKCE
!git pull

In [ ]:
# 3 · install deps
!pip install -q openml catboost optuna

In [ ]:
# 3b · OPTIONAL — only if OpenML 504s: upload openml_cache_clean5.tar.gz (else press Cancel)
import os, glob, tarfile
dst = '/root/.cache/openml/org/openml/www'
if glob.glob(dst + '/tasks/361055'):
    print('credit already cached — skip')
else:
    try:
        from google.colab import files; files.upload()
    except Exception as e: print('skipped:', e)
    hits = glob.glob('/content/**/openml_cache_clean5.tar.gz', recursive=True)
    if hits:
        os.makedirs(dst, exist_ok=True)
        with tarfile.open(hits[0]) as t: t.extractall(dst)
        print('cache extracted')
    else: print('no bundle — will use OpenML directly')

In [ ]:
# 4 · shared recipe (identical to the last two experiments)
base = ('--task 361055 --views x+tree --encoding oob --ensemble 2 --epochs 400 '
        '--dropout 0 --l1 0 --weight-decay 1e-3 --lr 3e-4 --batch-size 128 --device auto')
print(base)

In [ ]:
# 5 · PART A — the deletion ladder (12 arms, ~85 min)
!python -u run_fusion.py {base}                                                          --out results/fusion/dl_A00_control
# deepest first
!python -u run_fusion.py {base} --deep-frac 0.10          --deep-delete                  --out results/fusion/dl_A01_deep_10pct
!python -u run_fusion.py {base} --deep-frac 0.20          --deep-delete                  --out results/fusion/dl_A02_deep_20pct
!python -u run_fusion.py {base} --deep-layers 5           --deep-delete                  --out results/fusion/dl_A03_deep_L5
!python -u run_fusion.py {base} --deep-layers 4,5         --deep-delete                  --out results/fusion/dl_A04_deep_L45
!python -u run_fusion.py {base} --deep-layers 3,4,5       --deep-delete                  --out results/fusion/dl_A05_deep_L345
# random, matched sizes (the control)
!python -u run_fusion.py {base} --deep-select random --deep-frac 0.10    --deep-delete   --out results/fusion/dl_A06_rand_10pct
!python -u run_fusion.py {base} --deep-select random --deep-frac 0.20    --deep-delete   --out results/fusion/dl_A07_rand_20pct
!python -u run_fusion.py {base} --deep-select random --deep-layers 5     --deep-delete   --out results/fusion/dl_A08_rand_L5
!python -u run_fusion.py {base} --deep-select random --deep-layers 4,5   --deep-delete   --out results/fusion/dl_A09_rand_L45
!python -u run_fusion.py {base} --deep-select random --deep-layers 3,4,5 --deep-delete   --out results/fusion/dl_A10_rand_L345
# reverse arm: delete the SHALLOW bits instead
!python -u run_fusion.py {base} --deep-select shallowest --deep-layers 0,1,2 --deep-delete --out results/fusion/dl_A11_shallow_L012

In [ ]:
# 6 · PART B — depth-ramped bit flips vs matched uniform flips (4 arms, ~30 min)
!python -u run_fusion.py {base} --flip-ramp 0.2       --out results/fusion/dl_B01_ramp_020
!python -u run_fusion.py {base} --flip-ramp 0.5       --out results/fusion/dl_B02_ramp_050
!python -u run_fusion.py {base} --flip-uniform 0.16   --out results/fusion/dl_B03_unif_016
!python -u run_fusion.py {base} --flip-uniform 0.40   --out results/fusion/dl_B04_unif_040

In [ ]:
# 7 · summary table (works for whichever arms have finished)
import json, glob, pandas as pd, numpy as np
rows = []
for d in sorted(glob.glob('results/fusion/dl_*')):
    js = glob.glob(d + '/fusion_*.json')
    if not js: continue
    s = json.load(open(js[0])); r = s['results'][0]
    e = pd.read_csv(glob.glob(d + '/fusion_*_epochs.csv')[0])
    g = e.groupby('epoch').train_auc.mean()
    hit = g[g >= 0.99]
    if s.get('flip_ramp', 0) > 0:      kind, amt = 'ramp',    s['flip_mean_p']
    elif s.get('flip_uniform', 0) > 0: kind, amt = 'uniform', s['flip_mean_p']
    elif s.get('deep_delete'):         kind, amt = s['deep_select'], s['deep_frac_actual']
    else:                              kind, amt = 'control', 0.0
    rows.append(dict(arm=d.split('dl_')[-1], kind=kind,
                     removed_or_meanp=round(amt, 3),
                     n_bits=s.get('deep_n_bits', 0),
                     test_auc=r['test_auc'], best_val=r['best_val_auc'],
                     final_train_auc=round(g.iloc[-1], 4),
                     ep_train99=int(hit.index.min()) if len(hit) else None,
                     ceiling=s['tree_ceiling']))
t = pd.DataFrame(rows)
print(t.round(4).to_string(index=False))
c = t[t.kind=='control'].test_auc
if len(c):
    c = c.iloc[0]
    print(f'\ncontrol {c:.4f} | ceiling {t.ceiling.iloc[0]:.4f} | luck zone ~ +/-0.01')
    for _, r in t[t.kind!='control'].iterrows():
        print(f'  {r.arm:16s} vs control: {r.test_auc - c:+.4f}')
t.to_csv('results/fusion/dl_summary.csv', index=False)

In [ ]:
# 8 · the deletion curve + the noise bars
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [1.4, 1]})
a = ax[0]
if len(c):
    a.axhspan(c-0.01, c+0.01, color='grey', alpha=.12, label='luck zone (+/-0.01)')
    a.axhline(c, color='#3f5bd9', lw=1.5, label=f'control {c:.4f}')
a.axhline(t.ceiling.iloc[0], ls='--', color='#1f8a54', lw=1.5, label=f'tree ceiling {t.ceiling.iloc[0]:.4f}')
for kind, col, mk in [('deepest', '#cf3a4e', 'o'), ('random', '#8a92a1', 's')]:
    s_ = t[t.kind==kind].sort_values('removed_or_meanp')
    if len(s_): a.plot(s_.removed_or_meanp*100, s_.test_auc, mk+'-', color=col, ms=8, lw=2, label=f'delete {kind}')
s_ = t[t.kind=='shallowest']
if len(s_): a.scatter(s_.removed_or_meanp*100, s_.test_auc, marker='*', s=260, color='#b5730f', zorder=5, label='delete SHALLOW layers 0-2')
a.set_xlabel('% of tree bits deleted'); a.set_ylabel('test AUC')
a.set_title('Part A: where does the signal live?', fontweight='bold'); a.legend(fontsize=9); a.grid(alpha=.3)
b = ax[1]
nz = t[t.kind.isin(['ramp', 'uniform'])]
if len(nz):
    cols = ['#cf3a4e' if k=='ramp' else '#8a92a1' for k in nz.kind]
    b.bar(range(len(nz)), nz.test_auc, color=cols)
    for i, v in enumerate(nz.test_auc): b.text(i, v+0.0005, f'{v:.4f}', ha='center', fontsize=9)
    b.set_xticks(range(len(nz))); b.set_xticklabels([f'{k}\nmean p={p:.2f}' for k, p in zip(nz.kind, nz.removed_or_meanp)], fontsize=9)
    if len(c): b.axhline(c, color='#3f5bd9', lw=1.5); b.axhspan(c-0.01, c+0.01, color='grey', alpha=.12)
    b.set_ylim(min(nz.test_auc.min(), c)-0.02, max(nz.test_auc.max(), c)+0.01)
b.set_title('Part B: ramped vs uniform flips (same noise budget)', fontweight='bold'); b.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig('results/fusion/dl_overview.png', dpi=150); plt.show()

In [ ]:
# 9 · download everything
import shutil
from google.colab import files
shutil.make_archive('deletion_ladder_credit', 'zip', 'results/fusion')
files.download('deletion_ladder_credit.zip')